In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap 

# --- CONFIGURATION ---
NETWORK_FILE = "rules_pruned_x.txt" 
DATA_CSV = os.path.expanduser("~/TNBC_GSM158xxxx_BOOLEAN_p95.csv")
OUTPUT_EXCEL = "network_trajectory_highlighted_x.xlsx"

# ==========================================================
# INSERT HYBRID METHOD SUGGESTED NODES HERE
# ==========================================================
THERAPEUTIC_PERTURBATIONS = {'TRIP13': 0, 'BRCA1': 1, 'IKBKG': 1} 

# ==========================================================
# TARGET LIST (Same used in the hybrid method)
# ==========================================================
TARGET_LIST = {
    'APAF1': 1, 'BAD': 1, 'BCL2': 0, 'BCL2A1': 0, 'BCL2L1': 0,
    'BCL2L11': 1, 'BID': 1, 'BIRC2': 0, 'BIRC5' : 0, 'CASP10': 1,
    'CASP2': 1, 'CASP3': 1, 'CASP6': 1, 'CASP7': 1, 'CASP8': 1,
    'CASP9': 1, 'CYCS' : 1, 'DIABLO': 1, 'FADD': 1, 'MAP3K5': 1,
    'PMAIP1': 1, 'TRADD': 1, 'XIAP': 0, 'BAK1': 1, 'BAX': 1, 'MCL1': 0
}

print("⚡ STARTING STRICT VERIFICATION SIMULATION...")

# --- 1. RULES PARSING ---
rules = {}
all_genes = set()
BLACKLIST = {"False", "True", "0", "1", "and", "or", "not", "(", ")"}

if not os.path.exists(NETWORK_FILE):
    print(f"❌ ERROR: File '{NETWORK_FILE}' not found.")
else:
    with open(NETWORK_FILE, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"): continue
            if "*=" in line:
                target, rule = line.split("*=", 1)
                target, rule = target.strip(), rule.strip()
                rules[target] = rule
                all_genes.add(target)

    for r in rules.values():
        clean_r = r.replace("(", " ").replace(")", " ").replace(" not ", " ").replace(" and ", " ").replace(" or ", " ")
        for w in clean_r.split():
            if w not in BLACKLIST and not w.isdigit():
                all_genes.add(w)
    
    print(f"✅ Rules loaded. Total unique genes: {len(all_genes)}")

# --- 2. INITIAL STATE (STRICT) ---
state = {g: 0 for g in all_genes}
if os.path.exists(DATA_CSV):
    print("    Loading patient data from CSV...")
    df_initial = pd.read_csv(DATA_CSV)
    initial_map = dict(zip(df_initial['GeneSymbol'], df_initial['Boolean']))
    for g in all_genes:
        val = initial_map.get(g, None)
        if val is not None:
            state[g] = 1 if str(val).lower() in ['1', '1.0', 'true'] else 0
else:
    print("⚠️ Data CSV not found. Starting from all-zero state.")

# --- DRUG APPLICATION AT t=0 ---
for gene, value in THERAPEUTIC_PERTURBATIONS.items():
    if gene in state:
        state[gene] = value
        print(f"💉 Drug applied at t=0: {gene} forced to {value}")

# --- 3. DYNAMIC SIMULATION ---
print("\n--- ⚡ Evolution Calculation (Synchronous) ---")
trajectory = [state.copy()]
history_signatures = [str(list(state.values()))]
attractor_start_index = -1

for t in range(1, 501):
    next_state = {}
    for gene in all_genes:
        if gene in THERAPEUTIC_PERTURBATIONS:
            next_state[gene] = THERAPEUTIC_PERTURBATIONS[gene]
        elif gene in rules:
            try:
                val = eval(rules[gene], {}, state)
                next_state[gene] = 1 if val else 0
            # ========================================================
            # RIGOROUS TESTING: No exceptions are suppressed
            # ========================================================
            except Exception as e:
                print(f"\n❌ FATAL ERROR ON GENE: {gene}")
                print(f"Equation: {rules[gene]}")
                print(f"Python Error: {e}")
                print("⚠️ System frozen: Genes causing numerical computation failure are preventing model progression.")
                exit()
            # ========================================================
        else:
            next_state[gene] = state.get(gene, 0)

    state = next_state
    signature = str(list(state.values()))
    
    if signature in history_signatures:
        attractor_start_index = history_signatures.index(signature)
        print(f"🏁 Attractor reached at step t={t}")
        cycle_len = t - attractor_start_index
        print(f"   Type: {'✅ Stable Fixed Point' if cycle_len == 1 else f'❌ Oscillatory Cycle (Length {cycle_len})'}")
        break
    
    trajectory.append(state.copy())
    history_signatures.append(signature)

# --- 4. OUTPUT & EXCEL HIGHLIGHTING ---
df_traj = pd.DataFrame(trajectory)

# Sort columns alphabetically for Excel
df_traj = df_traj.reindex(sorted(df_traj.columns), axis=1)

df_traj.index.name = "TimeStep"

print(f"\n💾 Saving highlighted Excel file: {OUTPUT_EXCEL}")
try:
    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
        df_traj.to_excel(writer, sheet_name='Trajectory')
        workbook  = writer.book
        worksheet = writer.sheets['Trajectory']
        
        from openpyxl.styles import PatternFill
        yellow_fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
        green_fill = PatternFill(start_color='00FF00', end_color='00FF00', fill_type='solid') 
        
        start_row = attractor_start_index + 2
        end_row = len(df_traj) + 1
        
        fill_color = green_fill if cycle_len == 1 else yellow_fill

        for row in range(start_row, end_row + 1):
            for col in range(1, len(df_traj.columns) + 2):
                worksheet.cell(row=row, column=col).fill = fill_color
except ImportError:
    print("⚠️ 'openpyxl' library not found. Saving as CSV instead.")
    df_traj.to_csv("network_trajectory.csv")

# --- 5. DISCRETE HEATMAP (TARGET FOCUS) ---
print("\n📊 Generating Target Kinematic Heatmap...")

target_names = [g for g in TARGET_LIST.keys() if g in df_traj.columns]
target_names.sort() 

if len(target_names) > 0:
    time_limit = min(len(df_traj), attractor_start_index + 15)
    df_plot = df_traj[target_names].iloc[:time_limit]
    
    plt.figure(figsize=(16, 12))
    
    my_colors = ['#34495E', '#2ECC71'] 
    cmap_discrete = ListedColormap(my_colors)
    
    ax = sns.heatmap(
        df_plot.T, 
        cmap=cmap_discrete, 
        cbar_kws={'ticks': [0.25, 0.75], 'label': 'Logical State'}, 
        linewidths=0.5,
        linecolor='#2c3e50' 
    )
    
    cbar = ax.collections[0].colorbar
    cbar.set_ticklabels(['OFF (0)', 'ON (1)'])
    
    if attractor_start_index != -1:
        ax.axvline(x=attractor_start_index, color='#e74c3c', linewidth=4, linestyle='--')
        
        ax.text(attractor_start_index + 0.5, -0.5, "🏁 ATTRACTOR ENTRY", 
                color='#e74c3c', fontsize=14, fontweight='bold', ha='left', va='bottom')
                
        ax.axvspan(0, attractor_start_index, color='black', alpha=0.15)
        
        ax.text((attractor_start_index + time_limit)/2, len(target_names) + 0.8, 
                "STABILIZED ZONE", color='#27ae60', fontsize=12, fontweight='bold', ha='center')

    plt.title("Temporal Evolution of Target Nodes Towards Apoptosis", fontsize=18, fontweight='bold', pad=30)
    plt.xlabel("Time (Simulation Steps)", fontsize=14, fontweight='bold')
    plt.ylabel("Target Genes (Alphabetical Order)", fontsize=14, fontweight='bold')
    
    plt.yticks(rotation=0, fontsize=11) 
    plt.xticks(fontsize=10)
    
    plt.tight_layout()
    plot_filename = "Heatmap_Target_Attractor_41.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"✅ Masterpiece plot saved as: {plot_filename}")
    plt.show()
else:
    print("⚠️ Cannot generate heatmap: no target genes found.")